In [ ]:
from neo4j import GraphDatabase
from google import genai
import os
from dotenv import load_dotenv

In [ ]:
main_driver = GraphDatabase.driver(uri=os.getenv("NEO4J_URI"),auth=(os.getenv("NEO4J_USER"),os.getenv("NEO4J_PASS")))

client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))
MODEL = "gemini-2.5-flash"

In [6]:
with open("../prompts/graph_retrieval.txt", "r") as file:
    content = file.read()


def generate_cypher(query):
    prompt = content + query
    
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    
    return response.text.strip()

In [7]:
def run_cypher(cypher):
    with main_driver.session() as session:
        result = session.run(cypher)
        return [record.data() for record in result]

In [8]:
def test_query(user_query):
    print("\n🔍 User Query:", user_query)

    cypher = generate_cypher(user_query)

    try:
        results = run_cypher(cypher)
        print("\n📊 Results:")
        for r in results:
            print(r)
    except Exception as e:
        print("❌ Error executing query:", e)

In [9]:
test_query("What are the symptoms that can be relieved by Tulsi")


🔍 User Query: What are the symptoms that can be relieved by Tulsi

📊 Results:
{'s.name': 'Fatigue'}
{'s.name': 'High Cholesterol'}
{'s.name': 'High Blood Pressure'}
{'s.name': 'Common Cold'}
{'s.name': 'Cough'}
{'s.name': 'Respiratory Infections'}
{'s.name': 'Stress And Anxiety'}
{'s.name': 'Digestive Discomfort'}
{'s.name': 'Diabetes Support'}
{'s.name': 'Skin Infections'}
